In [ ]:
# UMUD Challenge: Muscle Architecture in Ultrasound Data
# Objective: To estimate the skeletal muscle architecture, namely pennal agnle (PA), fascicle length (FL) and muscle thickness (MT) from ultrasound images

In [ ]:
# import packages
import numpy as np
import pandas as pd
import torch
import torch.nn as nn   
import torch.nn.functional as F
from torchvision import transforms, datasets
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import os
from skimage import data, io, filters

In [ ]:
# import datasets
from PIL import Image
import cv2

shape = (256, 256)  # Resize images to 256x256

# create directory
base_dir = "/Users/helen1/Desktop/Kaggle CaseComp/UMUD Muscle Architecture_Sep26/apo_training_v1"
images_dir = os.path.join(base_dir, "apo_images_new_model_v1")
masks_dir = os.path.join(base_dir, "apo_masks_new_model_v1")

# get sorted list of files names
filenames = sorted([f for f in os.listdir(images_dir) if f.lower().endswith(('.tif', '.tiff'))])

images = []
masks = []

for fname in filenames:
    img_path = os.path.join(images_dir, fname)
    mask_path = os.path.join(masks_dir, fname)

    if not os.path.exists(mask_path):
        print(f"Warning: no matching mask for {fname}, skipping.")
        continue

    img = np.array(Image.open(img_path))
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # convert BGR to RGB if needed
    # make sure images are always 1 channel gray scale
    img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if img.ndim == 3 else img
    mask = np.array(Image.open(mask_path))
    if mask.ndim == 3:
        mask = mask[...,0]

    # resize images and masks to the same shape
    img = cv2.resize(img, shape, interpolation =cv2.INTER_LINEAR)
    mask = cv2.resize(mask, shape, interpolation =cv2.INTER_NEAREST)

    images.append(img)
    masks.append(mask)

# stack, normalise images and binarise masks to be only 0 and 1
images = np.stack(images).astype(np.float32)
images = images / 255.0  # rescale from 0-255 to 0-1

masks = np.stack(masks)
masks = (masks > 0).astype(np.float32)  # binarise masks to be only 0 and 1

# add channel dimension -> (N, H, W, 1)
images = images[..., np.newaxis]
masks = masks[..., np.newaxis]

print(images.shape, masks.shape)   # expect (1047, 256, 256, 1) for both

# loaded 1047 image/mask pairs



In [ ]:
# check label for mask images
#print("dtype:", images.dtype)
print("shape:", masks.shape)
print("min / max:", masks.min(), masks.max())
print("unique values:", np.unique(masks))

In [ ]:
import matplotlib.pyplot as plt

def show_pairs(images, masks, n=5, seed=None):
    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(images), size=min(n, len(images)), replace=False)

    fig, axes = plt.subplots(len(idxs), 3, figsize=(10, 3.3 * len(idxs)))
    axes = np.atleast_2d(axes)

    for row, i in enumerate(idxs):
        img, mask = images[i], masks[i]

        axes[row, 0].imshow(img, cmap="gray" if img.ndim == 2 else None)
        axes[row, 0].set_title(f"Image #{i}")

        axes[row, 1].imshow(mask, cmap="gray")
        axes[row, 1].set_title(f"Mask #{i}")

        # Overlay: mask drawn semi-transparently on top of the image
        axes[row, 2].imshow(img, cmap="gray" if img.ndim == 2 else None)
        axes[row, 2].imshow(np.ma.masked_where(mask == 0, mask), cmap="autumn", alpha=0.5)
        axes[row, 2].set_title("Overlay")

        for ax in axes[row]:
            ax.axis("off")

    plt.tight_layout()
    plt.show()

show_pairs(images, masks, n=5)

In [ ]:
# Build the U-net model, so we could train a model to predict masks from the images
# reference code: https://www.kaggle.com/code/yousseftaoudi/ultrasound-biomedical-image-segmentation-u-net
# The architecture of this U-net consists of a contracting path to capture context and a symmetric expanding path that enables precise localisation

from tensorflow.keras import datasets, layers, models
from tensorflow.keras.layers import BatchNormalization, Input,UpSampling2D,Concatenate, MaxPooling2D,ReLU, Conv2D,Concatenate,Conv2DTranspose,Cropping2D,Dropout
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import AUC
from tensorflow.keras.optimizers import SGD

# define a global bnb_relu function to add batch normalization layer after each convolutional layer to improve training stability and convergence
def bn_relu(x):
    x = BatchNormalization()(x)
    return ReLU()(x)

class U_Net():
    def __init__(self, inp=(572, 572, 1), min_filter=64, max_filter=1024, optimizer='Adam',loss=BinaryCrossentropy(), metrics = ['accuracy',AUC()]):
        self.max_filter = max_filter
        self.min_filter = min_filter
        self.img_inp = Input(shape = inp)
        down,cac,down_list = self.downsample()
        lower,down_list = self.contracting_path(down,down_list)
        upper,cac2 = self.upsample(lower,down_list)
        out = Conv2D(1,(1), padding = "same", activation="sigmoid", name='output')(upper)
        self.out = out
        self.model = keras.Model(inputs=self.img_inp, outputs=self.out)
        
        optimizer = tf.keras.optimizers.SGD(
            learning_rate=0.01, momentum=0.9,  name="SGD"
        )
        self.model.compile(optimizer=optimizer,loss=loss,metrics=metrics)


    def downsample(self):
        k = self.min_filter

        down_list = []
        res = self.img_inp
        while k < self.max_filter:
            res = Conv2D(k,(3,3), padding = "same")(res)
            res = bn_relu(res)
            #rest = Dropout(0.3)(res)
            res = Conv2D(k,(3,3), padding = "same")(res)
            res = bn_relu(res)
            #rest = Dropout(0.3)(res)
            down_list.append(res)
            cac = res
            res = MaxPooling2D(pool_size=(2, 2),strides=2)(res)
            curr = res
            k*=2
        return res,cac,down_list    

    def upsample(self,res,down_list):
        k = self.max_filter
        i=0
        while k > self.min_filter:
            res = Conv2D(k//2,(3,3), padding = "same")(res)
            res = bn_relu(res)
            #rest = Dropout(0.3)(res)
            k//=2
            
            if k > self.min_filter:
                res = Conv2D(k//2,(3,3), padding = "same")(res)
                res = bn_relu(res)
                #rest = Dropout(0.3)(res)
                res = UpSampling2D((2,2))(res)
                left = down_list.pop()
                res = self.crop_and_conc(left,res)
                i+=1
            else:
                res = Conv2D(k,(3,3), padding = "same")(res)
                res = bn_relu(res)
                #rest = Dropout(0.3)(res)
            cac = res
            curr = res
        return res,cac

    def contracting_path(self,down,down_list):
        res = Conv2D(self.max_filter,(3,3), padding = "same")(down)
        res = bn_relu(res)
        #rest = Dropout(0.3)(res)
        res = Conv2D(self.max_filter//2,(3,3), padding = "same")(res)
        res = bn_relu(res)
        #rest = Dropout(0.3)(res)
        res = UpSampling2D((2,2))(res)
        left = down_list.pop()
        res = self.crop_and_conc(left,res)
        return res,down_list
    
    
    def crop_and_conc(self,left,right):
        left = Cropping2D(cropping=int((left.shape[1]-right.shape[1])/2))(left)
        right = Concatenate()([left,right])
        return right

# check model is working
model = U_Net(inp=(256, 256, 1))



In [ ]:
# use images and masks to make a tf.data.Dataset object to feed into U-net model for training
# first transform the images and masks into numpy arrays, then create a tf.data.Dataset object from them
images = np.array(images, dtype="float32")
masks = np.array(masks, dtype="float32")

print(images.shape, masks.shape)

# split our 1047 images into training and validation sets, using 80% for training and 20% for validation
from sklearn.model_selection import train_test_split

x_train, x_val, y_train, y_val = train_test_split(
    images, masks, test_size=0.2, random_state=42
)

# now build the tf.data pipeline for both training and validation datasets
batch_size = 32
autotune = tf.data.AUTOTUNE

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(buffer_size=len(x_train)) # so to randomly order each epoch
    .batch(batch_size)
    .prefetch(autotune)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(batch_size)
    .prefetch(autotune)
)

# verfiy:
for img_batch, mask_batch in train_ds.take(1):
    print("Image batch shape:", img_batch.shape)
    print("Mask batch shape:", mask_batch.shape)

for img_batch, mask_batch in val_ds.take(1):
    print("Image batch shape:", img_batch.shape)
    print("Mask batch shape:", mask_batch.shape)

In [ ]:
# now feed the training and validation datasets into the U-net model for training
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# because aponeurosis masks are thin lines, the data for each mask is highly imbalanced, therefore, instead of using BCE as loss, we define a dice loss that handles imbalance
def dice_loss(y_true, y_pred, smooth=1.0):
    y_true = tf.reshape(y_true, [tf.shape(y_true)[0], -1])
    y_pred = tf.reshape(y_pred, [tf.shape(y_pred)[0], -1])
    inter = tf.reduce_sum(y_true * y_pred, axis=1)
    denom = tf.reduce_sum(y_true, axis=1) + tf.reduce_sum(y_pred, axis=1)
    return 1 - tf.reduce_mean((2 * inter + smooth) / (denom + smooth))

def bce_dice_loss(y_true, y_pred):
    return tf.keras.losses.binary_crossentropy(y_true, y_pred) + dice_loss(y_true, y_pred)

# because we have introduced dice loss, define dice metri as an additional monitoring metric to evaluate model performance, used alongside AUC
def dice_coef(y_true, y_pred, smooth=1.0):
    y_true = tf.reshape(y_true, [tf.shape(y_true)[0], -1])
    y_pred = tf.reshape(y_pred, [tf.shape(y_pred)[0], -1])
    inter = tf.reduce_sum(y_true * y_pred, axis=1)
    denom = tf.reduce_sum(y_true, axis=1) + tf.reduce_sum(y_pred, axis=1)
    return tf.reduce_mean((2 * inter + smooth) / (denom + smooth))

# introduce early stopping to prevent overfitting
callbacks = [
    EarlyStopping(
        monitor='val_loss', # monitor the combined loss
        patience=4,
        restore_best_weights=True, # make sure to restore the best model weights after early stopping, otherwise the model will be left with the weights from the last epoch, which may not be the best
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', # monitor the validation loss
        factor=0.8,        # when triggered, multiply the current learning rate by 0.8
        patience=3,        # keep this lower than EarlyStopping's patience
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        'best_unet.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]


model.model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), 
    loss=bce_dice_loss,
    metrics=["accuracy", AUC(), dice_coef]
    )

history = model.model.fit(
    train_ds,
    epochs=10, #only set epochs to 10 to reduce run time, for training purposes, epochs should be much higher
    validation_data=val_ds,
    callbacks=callbacks
)

In [ ]:
# for the aponeuroses masks, directly measure perpendicular distances between upper and lower line for muscle thickness (MT) 
from scipy import ndimage as ndi
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects, skeletonize

def line_distance(masks, pixel_spacing_mm=1.0, min_size=20): # make sure the measurement is in mm, not pixels
    masks = remove_small_objects(masks > 0, min_size=min_size)
    lab = label(masks)

    # keep the two largest components (the upper and lower aponeuroses lines)
    regions = sorted(regionprops(lab), key=lambda r: r.area, reverse=True)[:2]
    if len(regions) < 2:
        raise ValueError("Fewer than two lines found in mask")
    l1, l2 = (lab == regions[0].label), (lab == regions[1].label)
    # since the aponeuroses lines are very thick from the masks,  we need to skeletonize the lines first so we can more easily measure MT
    l1, l2 = skeletonize(l1), skeletonize(l2)

    # distance from every pixel of line 2 to the nearest pixel of line 1
    d = ndi.distance_transform_edt(~l1)[l2] * pixel_spacing_mm #convert from pixels to mm

    return {"mean": d.mean(), "median": np.median(d),
            "min": d.min(), "max": d.max()}

In [ ]:
# first check number of masks with less than 2 lines, as we cannot measure MT if there are fewer than 2 lines
def count_lines(mask, min_size=20):
    cleaned = remove_small_objects(mask > 0, min_size=min_size)
    return label(cleaned).max()

# masks: list of 2D arrays (or an N x H x W array); names: optional list of filenames
counts = [count_lines(m) for m in masks]

bad = [i for i, c in enumerate(counts) if c < 2]
print(f"{len(bad)} of {len(masks)} masks have fewer than two lines")
print("Indices:", bad)